# Nexora-MT v2 — Professional Offline English → Persian Subtitle Translator

Trains **Nexora-MT v2** (~55M parameters, Deep-Encoder / Fast-Decoder Transformer: `d_model=512`, `8 heads`, `6 enc / 4 dec`, `ffn_dim=1536`, `24k ByteLevel BPE`) on multi-source conversational & subtitle English→Persian corpora, then exports a single **Per-Row INT8 `.nxmt` v2** binary (~54 MB) with >99% FP32 accuracy for Nexora's pure-Rust CPU engine.

### Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (checkpoints auto-save after every epoch to `MyDrive/NexoraMT_v2/` so you can safely resume if Colab disconnects).
3. When finished, share `MyDrive/NexoraMT_v2/nexora-mt-en-fa-v2.nxmt` (or upload it to GitHub Releases).


In [ ]:
# Step 1: Verify GPU & Mount Google Drive for automatic checkpoint resume
!nvidia-smi
import os
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/NexoraMT_v2'
os.makedirs(WORK_DIR, exist_ok=True)
print('Workspace ready:', WORK_DIR)


In [ ]:
# Step 2: Install lightweight dependencies
!pip -q install --upgrade datasets tokenizers tqdm
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Please enable GPU: Runtime -> Change runtime type -> T4 GPU'


In [ ]:
# Step 3: Load, combine, normalize & filter multi-source EN->FA subtitle/conversational corpora
import re, random
from datasets import load_dataset

PERSIAN_RE = re.compile(r'[؀-ۿ]')
EN_RE = re.compile(r'[A-Za-z]')
HTML_TAG_RE = re.compile(r'<[^>]+>')

def normalize_persian(s: str) -> str:
    # Unify Arabic yeh/kaf to Persian standard & clean extra whitespace
    s = s.replace('ي', 'ی').replace('ك', 'ک')
    s = HTML_TAG_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s or '').strip()
    return s

def clean_en(s: str) -> str:
    s = HTML_TAG_RE.sub(' ', s or '')
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def is_good_pair(en: str, fa: str) -> bool:
    if not (2 <= len(en) <= 200 and 2 <= len(fa) <= 240):
        return False
    en_w = en.split()
    fa_w = fa.split()
    if not (1 <= len(en_w) <= 36 and 1 <= len(fa_w) <= 42):
        return False
    if not EN_RE.search(en) or not PERSIAN_RE.search(fa):
        return False
    ratio = len(fa) / max(len(en), 1)
    if ratio < 0.35 or ratio > 2.8:
        return False
    # Reject URLs or noisy markup
    if 'http://' in en or 'https://' in en or 'www.' in en:
        return False
    return True

pairs = []
seen = set()

def add_pair(en_raw, fa_raw, weight=1):
    en = clean_en(en_raw)
    fa = normalize_persian(fa_raw)
    if not is_good_pair(en, fa):
        return 0
    key = (en.lower(), fa)
    if key in seen:
        return 0
    seen.add(key)
    for _ in range(weight):
        pairs.append((en, fa))
    return 1

# Source 1: Helsinki-NLP/opus-100 (broad EN-FA base)
try:
    print('Loading Helsinki-NLP/opus-100 (en-fa)...')
    ds = load_dataset('Helsinki-NLP/opus-100', 'en-fa', split='train')
    c = 0
    for row in ds:
        tr = row.get('translation', {})
        c += add_pair(tr.get('en', ''), tr.get('fa', ''), weight=1)
    print(f'  + {c:,} clean pairs from opus-100')
except Exception as e:
    print('  opus-100 skipped:', e)

# Source 2: Conversational & TED/QED/Tatoeba corpora (2x weight for natural spoken style!)
EXTRA_DATASETS = [
    ('persiannlp/parsinlu_translation_en_fa', None, 'train', 'source', 'targets'),
    ('Helsinki-NLP/tatoeba_mt', 'eng-fas', 'test', 'sourceString', 'targetString'),
    ('Helsinki-NLP/tatoeba_mt', 'eng-fas', 'validation', 'sourceString', 'targetString'),
    ('qanastek/TED2020-en-fa', None, 'train', 'en', 'fa'),
]

for ds_name, cfg_name, split_name, en_col, fa_col in EXTRA_DATASETS:
    try:
        print(f'Loading {ds_name} ({cfg_name or "default"} / {split_name})...')
        ds = load_dataset(ds_name, cfg_name, split=split_name) if cfg_name else load_dataset(ds_name, split=split_name)
        c = 0
        for row in ds:
            en_v = row.get(en_col, '')
            fa_v = row.get(fa_col, '')
            if isinstance(fa_v, list):
                fa_v = fa_v[0] if fa_v else ''
            c += add_pair(str(en_v), str(fa_v), weight=2)
        print(f'  + {c:,} conversational pairs (weighted 2x) from {ds_name}')
    except Exception as e:
        print(f'  {ds_name} skipped ({e})')

print(f'Total training pool: {len(pairs):,} pairs ({len(seen):,} unique).')
random.seed(42)
random.shuffle(pairs)
VAL_SIZE = min(5000, max(1500, len(pairs) // 50))
val_pairs = pairs[:VAL_SIZE]
train_pairs = pairs[VAL_SIZE:]
print(f'Train: {len(train_pairs):,} | Val: {len(val_pairs):,}')


In [ ]:
# Step 4: Train 24,000-vocab Bilingual ByteLevel BPE Tokenizer (Nexora-BPE v2)
import os
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

TOK_PATH = os.path.join(WORK_DIR, 'nexora-bpe-en-fa-v2.json')
VOCAB_SIZE = 24000
SPECIAL_TOKENS = ['<pad>', '<bos>', '<eos>', '<unk>']
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

if os.path.exists(TOK_PATH):
    tokenizer = Tokenizer.from_file(TOK_PATH)
    print('Loaded existing v2 tokenizer from Drive:', TOK_PATH)
else:
    tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
    )
    def text_iter():
        for en, fa in train_pairs[:600000]:
            yield en
            yield fa
    tokenizer.train_from_iterator(text_iter(), trainer=trainer)
    tokenizer.save(TOK_PATH)
    print('Trained & saved new v2 tokenizer:', TOK_PATH)

sample = tokenizer.encode('How to be confident even if you are not.')
print('Sample encode:', sample.tokens[:12], '-> decoded:', tokenizer.decode(sample.ids))


In [ ]:
# Step 5: Define Nexora-MT v2 Architecture (Deep-Encoder 6L / Fast-Decoder 4L, d_model=512, 8 heads)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

CFG = {
    'arch': 'nexora-mt-v2',
    'vocab_size': VOCAB_SIZE,
    'd_model': 512,
    'n_heads': 8,
    'enc_layers': 6,
    'dec_layers': 4,
    'ffn_dim': 1536,
    'max_len': 128,
    'dropout': 0.1,
    'pad_id': PAD_ID,
    'bos_id': BOS_ID,
    'eos_id': EOS_ID,
}

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()
        self.w_gate = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_up = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(self.drop(F.silu(self.w_gate(x)) * self.w_up(x)))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, q, k, v, attn_mask=None, is_causal=False):
        B, Tq, D = q.shape
        Tk = k.shape[1]
        q = self.q_proj(q).view(B, Tq, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(B, Tk, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(B, Tk, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, Tq, D)
        return self.o_proj(out)

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, ffn_dim, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        h = self.norm1(x)
        x = x + self.drop(self.self_attn(h, h, h, attn_mask=src_mask))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, ffn_dim, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mem, src_mask=None):
        h = self.norm1(x)
        x = x + self.drop(self.self_attn(h, h, h, is_causal=True))
        x = x + self.drop(self.cross_attn(self.norm2(x), mem, mem, attn_mask=src_mask))
        x = x + self.drop(self.ffn(self.norm3(x)))
        return x

class NexoraMT(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg
        d = cfg['d_model']
        self.tok_emb = nn.Embedding(cfg['vocab_size'], d, padding_idx=cfg['pad_id'])
        self.pos_emb = nn.Embedding(cfg['max_len'], d)
        self.drop = nn.Dropout(cfg['dropout'])
        self.encoder = nn.ModuleList([
            EncoderLayer(d, cfg['n_heads'], cfg['ffn_dim'], cfg['dropout'])
            for _ in range(cfg['enc_layers'])
        ])
        self.enc_norm = RMSNorm(d)
        self.decoder = nn.ModuleList([
            DecoderLayer(d, cfg['n_heads'], cfg['ffn_dim'], cfg['dropout'])
            for _ in range(cfg['dec_layers'])
        ])
        self.dec_norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg['vocab_size'], bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

    def encode(self, src_ids):
        B, T = src_ids.shape
        pos = torch.arange(T, device=src_ids.device).unsqueeze(0)
        x = self.drop(self.tok_emb(src_ids) * math.sqrt(self.cfg['d_model']) + self.pos_emb(pos))
        src_mask = (src_ids != self.cfg['pad_id']).unsqueeze(1).unsqueeze(2)
        for layer in self.encoder:
            x = layer(x, src_mask=src_mask)
        return self.enc_norm(x), src_mask

    def decode(self, tgt_ids, mem, src_mask):
        B, T = tgt_ids.shape
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0)
        x = self.drop(self.tok_emb(tgt_ids) * math.sqrt(self.cfg['d_model']) + self.pos_emb(pos))
        for layer in self.decoder:
            x = layer(x, mem, src_mask=src_mask)
        return self.lm_head(self.dec_norm(x))

    def forward(self, src_ids, tgt_in):
        mem, src_mask = self.encode(src_ids)
        return self.decode(tgt_in, mem, src_mask)

    @torch.no_grad()
    def translate(self, text: str, tok, device, max_new_tokens: int = 64) -> str:
        self.eval()
        ids = [self.cfg['bos_id']] + tok.encode(text).ids[:self.cfg['max_len'] - 2] + [self.cfg['eos_id']]
        src = torch.tensor([ids], dtype=torch.long, device=device)
        mem, src_mask = self.encode(src)
        out = [self.cfg['bos_id']]
        for _ in range(max_new_tokens):
            tgt = torch.tensor([out], dtype=torch.long, device=device)
            logits = self.decode(tgt, mem, src_mask)
            nxt = int(logits[0, -1].argmax().item())
            if nxt == self.cfg['eos_id']:
                break
            out.append(nxt)
        return tok.decode(out[1:], skip_special_tokens=True).strip()

device = torch.device('cuda')
model = NexoraMT(CFG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Nexora-MT v2 initialized: {n_params / 1e6:.2f}M parameters')


In [ ]:
# Step 6: Train Nexora-MT v2 (22 epochs, grad accum = 2 -> effective batch 256, auto-resume)
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class SubtitleDataset(Dataset):
    def __init__(self, pairs_list, tok, max_len=96):
        self.pairs = pairs_list
        self.tok = tok
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, fa = self.pairs[idx]
        src = [BOS_ID] + self.tok.encode(en).ids[:self.max_len - 2] + [EOS_ID]
        tgt = [BOS_ID] + self.tok.encode(fa).ids[:self.max_len - 2] + [EOS_ID]
        return src, tgt

def collate_fn(batch):
    src_max = max(len(x[0]) for x in batch)
    tgt_max = max(len(x[1]) for x in batch)
    src_t = torch.full((len(batch), src_max), PAD_ID, dtype=torch.long)
    tgt_t = torch.full((len(batch), tgt_max), PAD_ID, dtype=torch.long)
    for i, (s, t) in enumerate(batch):
        src_t[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        tgt_t[i, :len(t)] = torch.tensor(t, dtype=torch.long)
    return src_t, tgt_t

BATCH_SIZE = 128
GRAD_ACCUM = 2
EPOCHS = 22
LR = 5e-4

train_loader = DataLoader(SubtitleDataset(train_pairs, tokenizer), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(SubtitleDataset(val_pairs, tokenizer), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), weight_decay=0.01)
steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = min(2000, total_steps // 10)

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / max(1, warmup_steps)
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * prog))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda')
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

CKPT_PATH = os.path.join(WORK_DIR, 'nexora_mt_v2_latest.pt')
BEST_PATH = os.path.join(WORK_DIR, 'nexora_mt_v2_best.pt')

start_epoch = 0
best_val_loss = float('inf')
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'Resumed from epoch {start_epoch} (best_val_loss={best_val_loss:.4f})')

TEST_SENTENCES = [
    'Welcome to Nexora live subtitles!',
    'How to be confident even if you are not.',
    'Watch out, two enemies are flanking from the left!',
    'Honestly, I did not expect that boss fight to have a second phase.',
]

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for step_idx, (src, tgt) in enumerate(pbar):
        src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        with torch.amp.autocast('cuda'):
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1)) / GRAD_ACCUM
        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM
        if (step_idx + 1) % GRAD_ACCUM == 0 or (step_idx + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        pbar.set_postfix(loss=f'{loss.item() * GRAD_ACCUM:.3f}')

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            with torch.amp.autocast('cuda'):
                logits = model(src, tgt[:, :-1])
                val_loss += criterion(logits.reshape(-1, VOCAB_SIZE), tgt[:, 1:].reshape(-1)).item()
    val_loss /= max(1, len(val_loader))
    print(f'Epoch {epoch+1} | train_loss={running_loss/len(train_loader):.4f} | val_loss={val_loss:.4f}')

    for s in TEST_SENTENCES:
        print('  EN:', s)
        print('  FA:', model.translate(s, tokenizer, device))

    state = {
        'epoch': epoch,
        'cfg': CFG,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': min(best_val_loss, val_loss),
    }
    torch.save(state, CKPT_PATH)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(state, BEST_PATH)
        print('  Saved new best v2 checkpoint to Drive!')


In [ ]:
# Step 7: Export Per-Row Symmetric INT8 Nexora-MT v2 binary (.nxmt v2) + SHA-256!
import hashlib, json, struct
import numpy as np

best_ckpt = torch.load(BEST_PATH if os.path.exists(BEST_PATH) else CKPT_PATH, map_location='cpu')
state_dict = best_ckpt['model']
with open(TOK_PATH, 'r', encoding='utf-8') as f:
    tok_json_str = f.read()

tensors_meta = []
payload_chunks = []
offset = 0

for name, tensor in state_dict.items():
    if name == 'lm_head.weight':
        continue  # weight-tied to tok_emb.weight
    arr = tensor.detach().float().numpy()
    if arr.ndim == 2:
        # Per-Row Symmetric INT8 quantization: stores f32[rows] scales + i8[rows, cols]
        row_max = np.max(np.abs(arr), axis=1, keepdims=True)
        scales = np.maximum(row_max / 127.0, 1e-8).astype(np.float32)
        q = np.clip(np.round(arr / scales), -127, 127).astype(np.int8)
        raw = scales.reshape(-1).tobytes() + q.tobytes()
        dtype = 'i8_row'
        scale_val = 1.0
    else:
        raw = arr.astype(np.float16).tobytes()
        dtype = 'f16'
        scale_val = 1.0
    tensors_meta.append({
        'name': name,
        'shape': list(arr.shape),
        'dtype': dtype,
        'scale': scale_val,
        'offset': offset,
        'nbytes': len(raw),
    })
    payload_chunks.append(raw)
    offset += len(raw)

header_obj = {
    'cfg': best_ckpt['cfg'],
    'tokenizer_json': tok_json_str,
    'tensors': tensors_meta,
}
header_bytes = json.dumps(header_obj, ensure_ascii=False).encode('utf-8')

OUT_NXMT = os.path.join(WORK_DIR, 'nexora-mt-en-fa-v2.nxmt')
with open(OUT_NXMT, 'wb') as f:
    f.write(b'NXMT')
    f.write(struct.pack('<II', 2, len(header_bytes)))
    f.write(header_bytes)
    for chunk in payload_chunks:
        f.write(chunk)

sha = hashlib.sha256(open(OUT_NXMT, 'rb').read()).hexdigest()
size_mb = os.path.getsize(OUT_NXMT) / (1024 * 1024)
print(f'Exported {OUT_NXMT} ({size_mb:.1f} MB)!')
print(f'SHA-256: {sha}')
